# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:31<00:00, 10.50s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

'Title: Samsung Galaxy Tab S10 Lite 10.9" Android Tablet for $310 + free shipping\nDetails: As one of its daily deals, Best Buy offers the Samsung Galaxy Tab S10 Lite 10.9" Android Tablet in Lite Gray or Coral Red for $310. That\'s a savings of $90 and at least $20 less than what you\'d pay elsewhere. Shipping is free.  Buy Now at Best Buy\nFeatures: 10.9" display 8GB RAM; 256GB Storage Model: SM-X400NZSMXAR\nURL: https://www.dealnews.com/products/Samsung/Samsung-Galaxy-Tab-S10-Lite-10-9-Android-Tablet/493728.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Certified Refurb RayNeo Air 4 Pro AR Glasses for $223 + free shipping
Details: Apply promo code "FAVEDEAL20" to get the refurb RayNeo Air 4 Pro AR Glasses for $223. It's the best price we could find by $76. Shipping is free. A 2-year Allstate warranty is included. Coupon expires June 14. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/Certified-Refurb-Ray-Neo-Air-4-Pro

In [9]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Certified refurbished RayNeo Air 4 Pro augmented-reality glasses that deliver a wearable AR experience through a lightweight frame. The headset supports immersive visual overlays for notifications, media, and compatible apps while maintaining a consumer-friendly design; the refurb unit includes a two-year warranty. Ideal for users who want hands-free AR functionality without buying new flagship smart glasses.', price=223.0, url='https://www.dealnews.com/Certified-Ref-Ray-Neo-Air-4-Pro-AR-Glasses-for-223-free-shipping/21839896.html?iref=rss-c142'), Deal(product_description='Samsung Q-Series HW-Q990F is an 11.1.4‑channel Dolby Atmos soundbar system that ships with a wireless subwoofer and separate satellite speakers to create a full-room immersive audio setup. It supports Alexa, Google Cast, and AirPlay 2, and includes features like Q‑Symphony for syncing with compatible Samsung TVs and advanced object-based audio processing for cinematic so

In [10]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Certified refurbished RayNeo Air 4 Pro augmented-reality glasses that deliver a wearable AR experience through a lightweight frame. The headset supports immersive visual overlays for notifications, media, and compatible apps while maintaining a consumer-friendly design; the refurb unit includes a two-year warranty. Ideal for users who want hands-free AR functionality without buying new flagship smart glasses.
223.0
https://www.dealnews.com/Certified-Ref-Ray-Neo-Air-4-Pro-AR-Glasses-for-223-free-shipping/21839896.html?iref=rss-c142

Samsung Q-Series HW-Q990F is an 11.1.4‑channel Dolby Atmos soundbar system that ships with a wireless subwoofer and separate satellite speakers to create a full-room immersive audio setup. It supports Alexa, Google Cast, and AirPlay 2, and includes features like Q‑Symphony for syncing with compatible Samsung TVs and advanced object-based audio processing for cinematic sound.
1000.0
https://www.dealnews.com/products/Samsung/Samsung-Q-Series-11-1-4-Channel-Sou

In [11]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [14]:
result

DealSelection(deals=[Deal(product_description='Refurbished RayNeo Air 4 Pro augmented-reality glasses offering a compact wearable AR experience. The Air 4 Pro packs AR optics and computing into a lightweight frame designed for hands-free notifications, media viewing, and basic spatial apps; this certified refurbished unit includes a two-year Allstate warranty and ships with tested hardware and firmware for reliable performance.', price=223.0, url='https://www.dealnews.com/Certified-Refurb-Ray-Neo-Air-4-Pro-AR-Glasses-for-223-free-shipping/21839896.html?iref=rss-c142'), Deal(product_description='Samsung Q-Series HW-Q990F is an 11.1.4-channel home theater soundbar system that includes a wired subwoofer and rear speakers to deliver immersive Dolby Atmos and DTS:X audio. It supports Q-Symphony for syncing with compatible Samsung TVs and modern connectivity like Alexa, Google Cast, and AirPlay 2, plus room-calibration features for optimized sound staging.', price=1000.0, url='https://www.de

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [15]:
load_dotenv(override=True)

True

In [16]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [17]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user not found
Pushover token not found


In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")